# Fongbe Challenge — Pipeline Kaggle (setup + fine-tuning w2v-bert-2.0)

1. Récupère CSV/scripts depuis GitHub, télécharge les tars `reliable` depuis le bucket GCS public, extrait, supprime les tars.
2. Fine-tune `facebook/w2v-bert-2.0` en CTC (~106 h).
3. Score sur voix tenues + sauvegarde les résultats en dataset/output Kaggle.

Runtime : **GPU T4** (Settings → Accelerator → GPU T4x2 ou P100).

## 0. Settings

In [ ]:
from pathlib import Path

DATA_DIR = Path('/kaggle/working/fongbe')
OUT_DIR  = Path('/kaggle/working/outputs')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_RAW = 'https://raw.githubusercontent.com/Appolinairee/fongbe-challenge/master/data'
GCS_BASE = 'https://storage.googleapis.com/fongbe'

# Archives à télécharger (reliable = ~13 GB WAV ; ajouter d'autres préfixes si besoin)
TAR_PREFIXES = ['audio_train_reliable_']

BASE_MODEL = 'facebook/w2v-bert-2.0'
MAX_STEPS = 4000
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
WARMUP_STEPS = 300
MAX_SECONDS = 15.0
N_VALIDATION = 300
SEED = 2026
print('settings ok')

## 1. CSV + scripts depuis GitHub

In [ ]:
import urllib.request
FILES = ['Train_reliable.csv','Train_uncertain.csv','Train_religious.csv','Test.csv',
         'SampleSubmission.csv','syllabify.py','normalize.py','score.py','test_score.py',
         'measure_frugality.py','conventions.md','README.md']
for name in FILES:
    dst = DATA_DIR / name
    if not dst.exists():
        urllib.request.urlretrieve(f'{REPO_RAW}/{name}', dst)
        print('✓', name)
print('data files ok:', len(list(DATA_DIR.glob('*.csv'))), 'CSV')

## 2. Télécharger + extraire les archives audio

Chaque tar est téléchargé, extrait dans `DATA_DIR`, puis **supprimé** pour rester dans le quota disque (~20 GB).

In [ ]:
import subprocess, shutil

TAR_DIR = DATA_DIR / 'tars'
TAR_DIR.mkdir(exist_ok=True)

# Inventaire du bucket (public)
import json, urllib.request
listing = json.loads(urllib.request.urlopen(
    'https://storage.googleapis.com/storage/v1/b/fongbe/o?maxResults=200&fields=items(name,size)').read())
all_tars = sorted(i['name'] for i in listing['items'] if i['name'].endswith('.tar'))
tars = [t for t in all_tars if any(t.startswith(p) for p in TAR_PREFIXES)]
print(f'{len(tars)} archives sélectionnées')

for name in tars:
    tar_path = TAR_DIR / name
    if not tar_path.exists():
        print(f'⬇️  {name}', flush=True)
        subprocess.run(['wget','-q','-O',str(tar_path),f'{GCS_BASE}/{name}'], check=True)
    print(f'📦 extraction {name}', flush=True)
    subprocess.run(['tar','-xf',str(tar_path),'-C',str(DATA_DIR)], check=True)
    tar_path.unlink()   # libère ~2 GB par archive
    shutil.rmtree(TAR_DIR / '.extracted', ignore_errors=True)
    print(f'✓ {name} extraite + tar supprimé', flush=True)

free = shutil.disk_usage('/kaggle/working').free/1e9
print(f'✅ Audio extrait. Disque libre : {free:.1f} GB')

In [ ]:
import pandas as pd
for group, csv_name, folder in [('reliable','Train_reliable.csv','audio/train_reliable'),
                                ('religious','Train_religious.csv','audio/train_religious'),
                                ('uncertain','Train_uncertain.csv','audio/train_uncertain'),
                                ('test','Test.csv','audio/test')]:
    sub = DATA_DIR / folder
    n = len(list(sub.glob('*.wav'))) if sub.exists() else 0
    print(f'{group:10s} {n:,} wav')
df = pd.read_csv(DATA_DIR/'Train_reliable.csv')
missing = (~df.audio_path.map(lambda p: (DATA_DIR/p).exists())).sum()
print(f'reliable: {len(df):,} lignes CSV, {missing} wav manquants')

In [ ]:
!pip install -q "transformers==5.14.1" soundfile "rapidfuzz==3.14.5"

In [ ]:
import sys, json, random
import numpy as np
import pandas as pd
sys.path.insert(0, str(DATA_DIR))
from normalize import tonal_target
from transformers import Wav2Vec2CTCTokenizer

random.seed(SEED); np.random.seed(SEED)

reliable = pd.read_csv(DATA_DIR / 'Train_reliable.csv')
reliable['target'] = reliable.transcription.map(tonal_target).str.strip()
reliable['wav'] = reliable.audio_path.map(lambda p: str(DATA_DIR / p))
available = reliable[reliable.wav.map(lambda p: Path(p).exists()) & (reliable.target != '')]

train_df = available[(available.recommended_validation == 0) & (available.duration <= MAX_SECONDS)]
valid_df = available[available.recommended_validation == 1]
valid_df = valid_df.sample(min(N_VALIDATION, len(valid_df)), random_state=SEED)
print(f'train: {len(train_df):,} utterances, {train_df.duration.sum()/3600:.1f} h, {train_df.speaker_id.nunique()} voices')
print(f'validation: {len(valid_df):,} utterances, {valid_df.speaker_id.nunique()} voices (unseen)')

chars = sorted(set(''.join(train_df.target)) - {' '})
vocab = {c: i for i, c in enumerate(chars)}
vocab['|'] = len(vocab); vocab['[UNK]'] = len(vocab); vocab['[PAD]'] = len(vocab)
(OUT_DIR / 'vocab.json').write_text(json.dumps(vocab, ensure_ascii=False))
tokenizer = Wav2Vec2CTCTokenizer(str(OUT_DIR / 'vocab.json'), unk_token='[UNK]', pad_token='[PAD]',
                                 word_delimiter_token='|', bos_token=None, eos_token=None)
print(len(vocab), 'symbols')

In [ ]:
import math, time
import soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (SeamlessM4TFeatureExtractor, Wav2Vec2BertProcessor, Wav2Vec2BertForCTC,
                          get_linear_schedule_with_warmup)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')
torch.manual_seed(SEED)

feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained(BASE_MODEL)
processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def load_audio(path):
    audio, sr = sf.read(path, dtype='float32')
    assert sr == 16000, path
    return audio if audio.ndim == 1 else audio.mean(axis=1)

class SpeechSet(Dataset):
    def __init__(self, df, with_labels=True):
        self.paths = df.wav.tolist()
        self.texts = df.target.tolist() if with_labels else None
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        item = {'input_features': feature_extractor(load_audio(self.paths[i]), sampling_rate=16000).input_features[0]}
        if self.texts is not None:
            item['labels'] = tokenizer(self.texts[i]).input_ids
        return item

def collate(items):
    batch = processor.pad([{'input_features': x['input_features']} for x in items], padding=True, return_tensors='pt')
    if 'labels' in items[0]:
        labels = processor.pad(labels=[{'input_ids': x['labels']} for x in items], padding=True, return_tensors='pt')
        batch['labels'] = labels['input_ids'].masked_fill(labels.attention_mask.ne(1), -100)
    return batch

def length_batches(durations, batch_size, shuffle):
    order = np.argsort(durations)
    batches = [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]
    if shuffle:
        random.shuffle(batches)
    return batches

model = Wav2Vec2BertForCTC.from_pretrained(
    BASE_MODEL, attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0, mask_time_prob=0.0,
    layerdrop=0.0, ctc_loss_reduction='mean', ctc_zero_infinity=True, add_adapter=True,
    pad_token_id=tokenizer.pad_token_id, vocab_size=len(tokenizer)).to(device)
model.gradient_checkpointing_enable()
print(f'{sum(p.numel() for p in model.parameters())/1e6:.0f} M parameters')

In [ ]:
train_set = SpeechSet(train_df)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)
scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, MAX_STEPS)
scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')

model.train()
step, micro, running, start = 0, 0, [], time.time()
while step < MAX_STEPS:
    loader = DataLoader(train_set, batch_sampler=length_batches(train_df.duration.values, BATCH_SIZE, True),
                        collate_fn=collate, num_workers=2)
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device == 'cuda'):
            loss = model(**batch).loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        running.append(loss.item() * GRAD_ACCUM)
        micro += 1
        if micro % GRAD_ACCUM:
            continue
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        step += 1
        if step % 100 == 0:
            print(f"step {step:5d}/{MAX_STEPS}  loss {np.mean(running):.3f}  {(time.time()-start)/60:5.1f} min")
            running = []
        if step >= MAX_STEPS:
            break

model.save_pretrained(OUT_DIR / 'model'); processor.save_pretrained(OUT_DIR / 'model')
print('saved in', OUT_DIR / 'model')

In [ ]:
import subprocess, sys

@torch.no_grad()
def transcribe(df, batch_size=8):
    model.eval()
    data = SpeechSet(df, with_labels=False)
    batches = length_batches(df.duration.values, batch_size, False)
    out = [''] * len(df)
    for idx in batches:
        batch = collate([data[i] for i in idx])
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device == 'cuda'):
            logits = model(input_features=batch['input_features'].to(device),
                           attention_mask=batch['attention_mask'].to(device)).logits
        for i, text in zip(idx, processor.batch_decode(logits.argmax(-1).cpu(), skip_special_tokens=True)):
            out[i] = text
    return out

valid_df = valid_df.copy()
valid_df['prediction'] = transcribe(valid_df)
valid_df[['id', 'transcription']].to_csv(OUT_DIR / 'valid_reference.csv', index=False)
valid_df[['id', 'prediction']].rename(columns={'prediction': 'transcription'}).to_csv(OUT_DIR / 'valid_prediction.csv', index=False)
print(subprocess.run([sys.executable, str(DATA_DIR / 'score.py'), '--reference', str(OUT_DIR / 'valid_reference.csv'),
                      '--submission', str(OUT_DIR / 'valid_prediction.csv')], capture_output=True, text=True).stdout)
for _, r in valid_df.head(5).iterrows():
    print('REF:', r.transcription, '\nHYP:', r.prediction, '\n')

In [ ]:
import shutil
from pathlib import Path
# Kaggle conserve /kaggle/working en output du notebook — copie dans outputs/final
DEST = Path('/kaggle/working/outputs/w2vbert_kaggle_run1')
DEST.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUT_DIR / 'model', DEST / 'model', dirs_exist_ok=True)
for f in ['vocab.json','valid_reference.csv','valid_prediction.csv']:
    if (OUT_DIR/f).exists(): shutil.copy2(OUT_DIR/f, DEST/f)
print('saved in', DEST)
